<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать  базовый  класс ShippingOption в  C#,  который  будет  представлять 
различные опции доставки. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void DeliveryStatusChangedHandler(string deliveryId, string oldStatus, string newStatus);
public delegate void CostCalculationHandler(decimal baseCost, decimal finalCost, string calculationType);
public delegate void NotificationEventHandler(string message, string recipient, DateTime timestamp);

public interface INotificationService
{
    void SendNotification(string message);
    void SendNotification(string message, string recipient);
    bool CanSendNotification();
}

public interface IDiscountService
{
    decimal CalculateDiscount(decimal baseCost);
    decimal CalculateDiscount(decimal baseCost, string promoCode);
    bool IsDiscountAvailable();
}

public class EmailNotificationService : INotificationService
{
    private string _smtpServer;
    private int _port;
    private bool _isConfigured;

    public string SmtpServer 
    { 
        get => _smtpServer;
        set => _smtpServer = !string.IsNullOrEmpty(value) ? value : "smtp.default.com";
    }

    public int Port 
    { 
        get => _port;
        set => _port = value > 0 ? value : 587;
    }

    public bool IsConfigured 
    { 
        get => _isConfigured;
        set => _isConfigured = value;
    }

    public EmailNotificationService(string smtpServer = "smtp.default.com", int port = 587)
    {
        SmtpServer = smtpServer;
        Port = port;
        IsConfigured = !string.IsNullOrEmpty(smtpServer);
    }

    public void SendNotification(string message)
    {
        if (CanSendNotification())
        {
            Console.WriteLine($"[EMAIL] Отправка уведомления: {message}");
            Console.WriteLine($"Через сервер: {SmtpServer}:{Port}");
        }
        else
        {
            Console.WriteLine("[EMAIL] Невозможно отправить уведомление - сервис не настроен");
        }
    }

    public void SendNotification(string message, string recipient)
    {
        if (CanSendNotification())
        {
            Console.WriteLine($"[EMAIL] Отправка уведомления для {recipient}: {message}");
            Console.WriteLine($"Через сервер: {SmtpServer}:{Port}");
        }
        else
        {
            Console.WriteLine("[EMAIL] Невозможно отправить уведомление - сервис не настроен");
        }
    }

    public bool CanSendNotification()
    {
        return IsConfigured && !string.IsNullOrEmpty(SmtpServer);
    }

    public void SetConfiguration(string smtpServer, int port)
    {
        SmtpServer = smtpServer;
        Port = port;
        IsConfigured = true;
        Console.WriteLine($"Конфигурация email обновлена: {smtpServer}:{port}");
    }

    public string GetConfiguration()
    {
        return $"SMTP: {SmtpServer}:{Port}, Настроен: {IsConfigured}";
    }

    public bool ValidateConfiguration()
    {
        return IsConfigured && Port > 0 && !string.IsNullOrEmpty(SmtpServer);
    }
}

public class SeasonalDiscountService : IDiscountService
{
    private decimal _seasonalDiscount;
    private DateTime _seasonStart;
    private DateTime _seasonEnd;

    public decimal SeasonalDiscount 
    { 
        get => _seasonalDiscount;
        set => _seasonalDiscount = value >= 0 && value <= 0.5m ? value : 0.1m;
    }

    public DateTime SeasonStart 
    { 
        get => _seasonStart;
        set => _seasonStart = value;
    }

    public DateTime SeasonEnd 
    { 
        get => _seasonEnd;
        set => _seasonEnd = value;
    }

    public SeasonalDiscountService(decimal discount = 0.1m)
    {
        SeasonalDiscount = discount;
        SeasonStart = new DateTime(DateTime.Now.Year, 1, 1);
        SeasonEnd = new DateTime(DateTime.Now.Year, 12, 31);
    }

    public decimal CalculateDiscount(decimal baseCost)
    {
        if (IsDiscountAvailable())
        {
            decimal discount = baseCost * SeasonalDiscount;
            Console.WriteLine($"Применена сезонная скидка: {SeasonalDiscount:P0} = {discount} руб.");
            return discount;
        }
        return 0;
    }

    public decimal CalculateDiscount(decimal baseCost, string promoCode)
    {
        decimal discount = CalculateDiscount(baseCost);
        
        if (!string.IsNullOrEmpty(promoCode) && promoCode.ToUpper() == "SUMMER2024")
        {
            decimal promoDiscount = baseCost * 0.05m;
            discount += promoDiscount;
            Console.WriteLine($"Дополнительная скидка по промокоду: 5% = {promoDiscount} руб.");
        }
        
        return discount;
    }

    public bool IsDiscountAvailable()
    {
        DateTime now = DateTime.Now;
        return now >= SeasonStart && now <= SeasonEnd;
    }

    public void SetSeason(DateTime start, DateTime end)
    {
        if (start < end)
        {
            SeasonStart = start;
            SeasonEnd = end;
            Console.WriteLine($"Сезон скидок установлен: {start:dd.MM.yyyy} - {end:dd.MM.yyyy}");
        }
        else
        {
            Console.WriteLine("Ошибка: дата начала должна быть раньше даты окончания");
        }
    }

    public bool IsSeasonActive()
    {
        return IsDiscountAvailable();
    }

    public int DaysUntilSeasonEnd()
    {
        if (DateTime.Now < SeasonEnd)
        {
            return (SeasonEnd - DateTime.Now).Days;
        }
        return 0;
    }
}

public class DependencyContainer
{
    private readonly Dictionary<Type, object> _services;

    public DependencyContainer()
    {
        _services = new Dictionary<Type, object>();
    }

    public void Register<T>(T service)
    {
        _services[typeof(T)] = service;
    }

    public T GetService<T>()
    {
        if (_services.ContainsKey(typeof(T)))
        {
            return (T)_services[typeof(T)];
        }
        throw new InvalidOperationException($"Сервис {typeof(T).Name} не зарегистрирован");
    }

    public bool IsRegistered<T>()
    {
        return _services.ContainsKey(typeof(T));
    }
}

public class Pickup : ShippingOption
{
    private string _pickupAddress;
    private int _storageDays;
    private TimeSpan _workingHoursStart;
    private TimeSpan _workingHoursEnd;
    private int _maxPackageSize;
    private bool _hasParking;
    private List<string> _paymentMethods;

    public string PickupAddress 
    { 
        get => _pickupAddress;
        set => _pickupAddress = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("Адрес не может быть пустым");
    }

    public int StorageDays 
    { 
        get => _storageDays;
        set => _storageDays = value >= 1 ? value : 7;
    }

    public TimeSpan WorkingHoursStart 
    { 
        get => _workingHoursStart;
        set => _workingHoursStart = value;
    }

    public TimeSpan WorkingHoursEnd 
    { 
        get => _workingHoursEnd;
        set => _workingHoursEnd = value;
    }

    public int MaxPackageSize 
    { 
        get => _maxPackageSize;
        set => _maxPackageSize = value > 0 ? value : 100;
    }

    public bool HasParking 
    { 
        get => _hasParking;
        set => _hasParking = value;
    }

    public List<string> PaymentMethods 
    { 
        get => _paymentMethods;
        set => _paymentMethods = value ?? new List<string> { "Наличные", "Карта" };
    }

    public Pickup(string id, string name, string address, int storageDays = 7) 
        : base(id, name, 0)
    {
        PickupAddress = address;
        StorageDays = storageDays;
        WorkingHoursStart = TimeSpan.FromHours(9);
        WorkingHoursEnd = TimeSpan.FromHours(21);
        MaxPackageSize = 100;
        HasParking = true;
        PaymentMethods = new List<string> { "Наличные", "Карта", "Онлайн-оплата" };
    }

    public void AddPaymentMethod(string method)
    {
        if (!string.IsNullOrEmpty(method) && !PaymentMethods.Contains(method))
        {
            PaymentMethods.Add(method);
            Console.WriteLine($"Добавлен способ оплаты: {method}");
            OnNotificationSent($"Добавлен способ оплаты: {method}", "Payment System", DateTime.Now);
        }
    }

    public bool SupportsPaymentMethod(string method)
    {
        return PaymentMethods.Contains(method, StringComparer.OrdinalIgnoreCase);
    }

    public void UpdateFacilityInfo(int maxSize, bool hasParking)
    {
        MaxPackageSize = maxSize;
        HasParking = hasParking;
        Console.WriteLine($"Информация об объекте обновлена: Макс. размер: {maxSize}см, Парковка: {(hasParking ? "есть" : "нет")}");
    }

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Самовывоз, Адрес: {PickupAddress}, Хранение: {StorageDays} дней";

    public override string EstimateDeliveryTime() => 
        $"Готов к выдаче через 2 часа, Время работы: {WorkingHoursStart:hh\\:mm}-{WorkingHoursEnd:hh\\:mm}";

    public override decimal CalculateCost() => 0;

    public override string GetDeliveryDetails(bool includeStatus, bool includePriority)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}";
        }
        if (includePriority)
        {
            details += $", Приоритет: {PriorityLevel}, Адрес: {PickupAddress}";
        }
        return details;
    }

    public bool IsWorkingNow()
    {
        var now = DateTime.Now.TimeOfDay;
        return now >= WorkingHoursStart && now <= WorkingHoursEnd;
    }

    public bool IsWorkingNow(DateTime customTime)
    {
        var time = customTime.TimeOfDay;
        return time >= WorkingHoursStart && time <= WorkingHoursEnd;
    }

    public DateTime GetExpiryDate()
    {
        return DateTime.Now.AddDays(StorageDays);
    }

    public DateTime GetExpiryDate(DateTime fromDate)
    {
        return fromDate.AddDays(StorageDays);
    }

    public override bool CanCombineWith(ShippingOption other) => 
        other is StandardDelivery;
}

public class ShippingOption
{
    public string DeliveryOptionId { get; set; }
    public string DeliveryOptionName { get; set; }
    public decimal Cost { get; set; }
    public DateTime CreatedDate { get; set; }
    public bool IsActive { get; set; }
    public int PriorityLevel { get; set; }
    public string ProviderName { get; set; }
    public int MaxDeliveryAttempts { get; set; }
    public bool RequiresSignature { get; set; }
    public string SecurityLevel { get; set; }
    public List<string> SupportedCountries { get; set; }
    public DateTime LastModifiedDate { get; set; }

    public event DeliveryStatusChangedHandler StatusChanged;
    public event CostCalculationHandler CostCalculated;
    public event NotificationEventHandler NotificationSent;

    protected virtual void OnStatusChanged(string deliveryId, string oldStatus, string newStatus)
    {
        StatusChanged?.Invoke(deliveryId, oldStatus, newStatus);
    }

    protected virtual void OnCostCalculated(decimal baseCost, decimal finalCost, string calculationType)
    {
        CostCalculated?.Invoke(baseCost, finalCost, calculationType);
    }

    protected virtual void OnNotificationSent(string message, string recipient, DateTime timestamp)
    {
        NotificationSent?.Invoke(message, recipient, timestamp);
    }

    protected readonly INotificationService _notificationService;
    protected readonly IDiscountService _discountService;

    public ShippingOption(string id, string name, decimal cost, 
                         INotificationService notificationService = null, 
                         IDiscountService discountService = null)
    {
        DeliveryOptionId = id;
        DeliveryOptionName = name;
        Cost = cost;
        CreatedDate = DateTime.Now;
        LastModifiedDate = DateTime.Now;
        IsActive = true;
        PriorityLevel = 3;
        ProviderName = "Standard Provider";
        MaxDeliveryAttempts = 3;
        RequiresSignature = false;
        SecurityLevel = "Standard";
        SupportedCountries = new List<string> { "Россия" };
        _notificationService = notificationService;
        _discountService = discountService;
    }

    public virtual void UpdateSupportedCountries(List<string> countries)
    {
        if (countries != null && countries.Any())
        {
            SupportedCountries = countries;
            LastModifiedDate = DateTime.Now;
            Console.WriteLine($"Обновлены поддерживаемые страны: {string.Join(", ", countries)}");
            OnNotificationSent($"Обновлены поддерживаемые страны для {DeliveryOptionName}", 
                                   "System Administrator", DateTime.Now);
        }
    }

    public virtual bool IsCountrySupported(string country)
    {
        return SupportedCountries.Contains(country, StringComparer.OrdinalIgnoreCase);
    }

    public virtual void ChangeSecurityLevel(string newLevel)
    {
        var oldLevel = SecurityLevel;
        SecurityLevel = newLevel;
        LastModifiedDate = DateTime.Now;
        Console.WriteLine($"Уровень безопасности изменен: {oldLevel} -> {newLevel}");
        OnStatusChanged(DeliveryOptionId, $"Security:{oldLevel}", $"Security:{newLevel}");
    }

    public virtual void SendCreationNotification()
    {
        _notificationService?.SendNotification($"Создана новая опция доставки: {DeliveryOptionName}");
        OnNotificationSent($"Создана новая опция доставки: {DeliveryOptionName}", 
                               "System", DateTime.Now);
    }

    public virtual decimal CalculateFinalCost()
    {
        decimal baseCost = CalculateCost();
        decimal discount = _discountService?.CalculateDiscount(baseCost) ?? 0;
        decimal finalCost = baseCost - discount;
        OnCostCalculated(baseCost, finalCost, "Final Cost Calculation");
        return finalCost;
    }

    public virtual string GetProviderInfo()
    {
        return $"Провайдер: {ProviderName}, Макс. попыток доставки: {MaxDeliveryAttempts}, Требуется подпись: {RequiresSignature}";
    }

    public virtual void UpdateProvider(string providerName, int maxAttempts, bool requiresSignature)
    {
        ProviderName = providerName;
        MaxDeliveryAttempts = maxAttempts;
        RequiresSignature = requiresSignature;
        Console.WriteLine($"Информация о провайдере обновлена: {GetProviderInfo()}");
    }

    public virtual bool ValidateConfiguration()
    {
        return !string.IsNullOrEmpty(DeliveryOptionId) && 
               !string.IsNullOrEmpty(DeliveryOptionName) && 
               Cost >= 0 && 
               !string.IsNullOrEmpty(ProviderName) &&
               MaxDeliveryAttempts >= 1;
    }

    public virtual string GetSecurityInfo()
    {
        return $"Требуется подпись: {RequiresSignature}, Попытки доставки: {MaxDeliveryAttempts}, Приоритет: {PriorityLevel}, Уровень безопасности: {SecurityLevel}";
    }
    
    public virtual decimal CalculateCost() => Cost;
    
    public virtual string EstimateDeliveryTime() => "Время доставки не указано";
    
    public virtual string GetDeliveryDetails() => 
        $"ID: {DeliveryOptionId}, Название: {DeliveryOptionName}, Стоимость: {Cost} руб.";

    public virtual string GetDeliveryDetails(bool includeStatus)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}";
        }
        return details;
    }

    public virtual string GetDeliveryDetails(bool includeStatus, bool includePriority)
    {
        var details = GetDeliveryDetails(includeStatus);
        if (includePriority)
        {
            details += $", Приоритет: {PriorityLevel}";
        }
        return details;
    }

    public virtual void UpdateCost(decimal newCost)
    {
        Cost = newCost;
        Console.WriteLine($"Стоимость обновлена: {newCost} руб.");
    }

    public virtual void UpdateCost(decimal newCost, string reason)
    {
        Cost = newCost;
        Console.WriteLine($"Стоимость обновлена: {newCost} руб. Причина: {reason}");
    }

    public virtual void ActivateOption()
    {
        var oldStatus = GetStatus();
        IsActive = true;
        Console.WriteLine($"Опция {DeliveryOptionName} активирована");
        OnStatusChanged(DeliveryOptionId, oldStatus, GetStatus());
    }

    public virtual void DeactivateOption()
    {
        var oldStatus = GetStatus();
        IsActive = false;
        Console.WriteLine($"Опция {DeliveryOptionName} деактивирована");
        OnStatusChanged(DeliveryOptionId, oldStatus, GetStatus());
    }

    public virtual string GetStatus()
    {
        return IsActive ? "Активна" : "Неактивна";
    }

    public virtual bool CanCombineWith(ShippingOption other) => false;
}

public class StandardDelivery : ShippingOption
{
    public int AverageDeliveryTime { get; set; }
    public int MaxWeight { get; set; }
    public string DeliveryRegion { get; set; }
    public bool HasTracking { get; set; }
    public string PackageType { get; set; }
    public decimal InsuranceCost { get; set; }
    public decimal WeightSurcharge { get; set; }
    public bool IsEcoFriendly { get; set; }
    public List<string> SpecialHandlingInstructions { get; set; }

    public StandardDelivery(string id, string name, decimal cost, int avgTime, 
                           INotificationService notificationService = null, 
                           IDiscountService discountService = null,
                           int maxWeight = 50, string region = "Все регионы") 
        : base(id, name, cost, notificationService, discountService)
    {
        AverageDeliveryTime = avgTime;
        MaxWeight = maxWeight;
        DeliveryRegion = region;
        HasTracking = true;
        PackageType = "Стандартная";
        InsuranceCost = cost * 0.01m;
        WeightSurcharge = 0;
        IsEcoFriendly = false;
        SpecialHandlingInstructions = new List<string>();
    }

    public void ApplyWeightSurcharge(int actualWeight)
    {
        if (actualWeight > MaxWeight)
        {
            WeightSurcharge = (actualWeight - MaxWeight) * 10m;
            Console.WriteLine($"Применена доплата за вес: {WeightSurcharge} руб.");
            OnCostCalculated(Cost, Cost + WeightSurcharge, "Weight Surcharge");
        }
    }

    public void AddHandlingInstruction(string instruction)
    {
        if (!string.IsNullOrEmpty(instruction))
        {
            SpecialHandlingInstructions.Add(instruction);
            Console.WriteLine($"Добавлена инструкция по обработке: {instruction}");
            OnNotificationSent($"Добавлена инструкция для {DeliveryOptionName}: {instruction}", 
                                   "Logistics Team", DateTime.Now);
        }
    }

    public void SetEcoFriendlyMode(bool enable)
    {
        IsEcoFriendly = enable;
        var mode = enable ? "активирован" : "деактивирован";
        Console.WriteLine($"Эко-режим {mode} для доставки {DeliveryOptionName}");
        if (enable)
        {
            AddHandlingInstruction("Использовать экологичную упаковку");
        }
    }

    public override decimal CalculateFinalCost()
    {
        decimal baseCost = CalculateCost();
        decimal discount = _discountService?.CalculateDiscount(baseCost) ?? 0;
        decimal total = baseCost - discount + InsuranceCost + WeightSurcharge;
        Console.WriteLine($"Итоговая стоимость: {baseCost} - {discount} + {InsuranceCost} (страховка) + {WeightSurcharge} (доплата за вес) = {total} руб.");
        OnCostCalculated(baseCost, total, "Standard Delivery Final Cost");
        return total;
    }

    public string GetTrackingInfo()
    {
        return HasTracking ? 
            $"Трек номер: STD-{DeliveryOptionId}-{DateTime.Now:yyyyMMdd}" : 
            "Отслеживание недоступно";
    }

    public void UpdatePackageInfo(string packageType, bool hasTracking, decimal insurancePercent)
    {
        PackageType = packageType;
        HasTracking = hasTracking;
        InsuranceCost = Cost * (insurancePercent / 100);
        Console.WriteLine($"Информация о посылке обновлена: Тип: {PackageType}, Отслеживание: {HasTracking}, Страховка: {InsuranceCost} руб.");
    }

    public override string GetProviderInfo()
    {
        return base.GetProviderInfo() + $", Тип упаковки: {PackageType}, Отслеживание: {HasTracking}, Эко-режим: {IsEcoFriendly}";
    }

    public decimal CalculateInsuranceCost(decimal declaredValue)
    {
        if (declaredValue > 0)
        {
            return declaredValue * 0.02m;
        }
        return InsuranceCost;
    }

    public bool RequiresSpecialHandling()
    {
        return PackageType == "Хрупкая" || MaxWeight > 30 || SpecialHandlingInstructions.Any();
    }

    public override string EstimateDeliveryTime() => 
        $"Среднее время: {AverageDeliveryTime} дней";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Стандартная, {EstimateDeliveryTime()}, Макс. вес: {MaxWeight}кг, Регион: {DeliveryRegion}";

    public override string GetDeliveryDetails(bool includeStatus)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}, Регион доставки: {DeliveryRegion}";
        }
        return details;
    }

    public bool CanDeliverTo(string region)
    {
        return DeliveryRegion == "Все регионы" || DeliveryRegion == region;
    }

    public bool CanDeliverTo(string region, int weight)
    {
        return CanDeliverTo(region) && weight <= MaxWeight;
    }

    public decimal CalculateWeightSurcharge(int weight)
    {
        if (weight > MaxWeight)
            return Cost * 0.1m * (weight - MaxWeight);
        return 0;
    }

    public decimal CalculateWeightSurcharge(int weight, decimal multiplier)
    {
        if (weight > MaxWeight)
            return Cost * multiplier * (weight - MaxWeight);
        return 0;
    }

    public override bool CanCombineWith(ShippingOption other) => 
        other is Pickup;
}

public class ExpressDelivery : ShippingOption
{
    public int MinDeliveryTime { get; set; }
    public decimal ExpressFee { get; set; }
    public bool GuaranteedDelivery { get; set; }
    public TimeSpan DeliveryTimeWindow { get; set; }
    public bool HasPriorityLoading { get; set; }
    public List<DateTime> AvailableDeliverySlots { get; set; }

    public ExpressDelivery(string id, string name, decimal cost, int minTime, decimal expressFee = 100, bool guaranteed = true) 
        : base(id, name, cost)
    {
        MinDeliveryTime = minTime;
        ExpressFee = expressFee;
        GuaranteedDelivery = guaranteed;
        DeliveryTimeWindow = TimeSpan.FromHours(4);
        HasPriorityLoading = true;
        AvailableDeliverySlots = new List<DateTime>();
        GenerateDeliverySlots();
    }

    private void GenerateDeliverySlots()
    {
        AvailableDeliverySlots.Clear();
        var now = DateTime.Now;
        for (int i = 0; i < 7; i++)
        {
            var date = now.AddDays(i + 1);
            AvailableDeliverySlots.Add(date.Date.AddHours(9));
            AvailableDeliverySlots.Add(date.Date.AddHours(13));
            AvailableDeliverySlots.Add(date.Date.AddHours(17));
        }
    }

    public void DisplayAvailableSlots()
    {
        Console.WriteLine($"Доступные слоты для экспресс-доставки {DeliveryOptionName}:");
        foreach (var slot in AvailableDeliverySlots)
        {
            Console.WriteLine($"  - {slot:dd.MM.yyyy HH:mm}");
        }
    }

    public bool BookDeliverySlot(DateTime preferredSlot)
    {
        var availableSlot = AvailableDeliverySlots.FirstOrDefault(slot => 
            slot.Date == preferredSlot.Date && 
            Math.Abs((slot - preferredSlot).TotalHours) <= 2);
        if (availableSlot != default(DateTime))
        {
            AvailableDeliverySlots.Remove(availableSlot);
            Console.WriteLine($"Слот забронирован: {availableSlot:dd.MM.yyyy HH:mm}");
            OnNotificationSent($"Забронирован слот доставки для {DeliveryOptionName} на {availableSlot:dd.MM.yyyy HH:mm}", 
                                   "Customer", DateTime.Now);
            return true;
        }
        Console.WriteLine("Подходящий слот не найден");
        return false;
    }

    public override decimal CalculateCost() => Cost * 1.5m + ExpressFee;

    public override string EstimateDeliveryTime() => 
        $"Минимальное время: {MinDeliveryTime} дней, Гарантия: {GuaranteedDelivery}";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Экспресс, {EstimateDeliveryTime()}, Итог: {CalculateCost()} руб.";

    public override void UpdateCost(decimal newCost)
    {
        base.UpdateCost(newCost);
        Console.WriteLine($"Внимание: Экспресс-доставка имеет дополнительную плату {ExpressFee} руб.");
    }

    public void ScheduleDelivery(DateTime deliveryDate)
    {
        Console.WriteLine($"Экспресс-доставка запланирована на {deliveryDate:dd.MM.yyyy}");
    }

    public void ScheduleDelivery(DateTime deliveryDate, string timeSlot)
    {
        Console.WriteLine($"Экспресс-доставка запланирована на {deliveryDate:dd.MM.yyyy} в интервал {timeSlot}");
    }

    public bool IsUrgentDelivery()
    {
        return MinDeliveryTime <= 1;
    }

    public bool IsUrgentDelivery(int customUrgentThreshold)
    {
        return MinDeliveryTime <= customUrgentThreshold;
    }

    public override bool CanCombineWith(ShippingOption other) => 
        !(other is ExpressDelivery);
}

public class DeliveryCollection<T> where T : ShippingOption
{
    private List<T> _items;

    public delegate bool DeliveryFilter(T delivery);
    
    public event Action<string, int> CollectionChanged;

    public DeliveryCollection()
    {
        _items = new List<T>();
    }

    public void AddItem(T item)
    {
        _items.Add(item);
        Console.WriteLine($"Добавлено в коллекцию: {item.DeliveryOptionName}");
        item.StatusChanged += OnDeliveryStatusChanged;
        item.CostCalculated += OnCostCalculated;
        item.NotificationSent += OnNotificationSent;
        CollectionChanged?.Invoke("Item Added", _items.Count);
    }

    public void RemoveItem(string id)
    {
        var item = _items.Find(x => x.DeliveryOptionId == id);
        if (item != null)
        {
            item.StatusChanged -= OnDeliveryStatusChanged;
            item.CostCalculated -= OnCostCalculated;
            item.NotificationSent -= OnNotificationSent;
            _items.Remove(item);
            Console.WriteLine($"Удалено из коллекции: {item.DeliveryOptionName}");
            CollectionChanged?.Invoke("Item Removed", _items.Count);
        }
    }

    private void OnDeliveryStatusChanged(string deliveryId, string oldStatus, string newStatus)
    {
        Console.WriteLine($"[КОЛЛЕКЦИЯ] Статус изменен: {deliveryId} - {oldStatus} -> {newStatus}");
    }

    private void OnCostCalculated(decimal baseCost, decimal finalCost, string calculationType)
    {
        Console.WriteLine($"[КОЛЛЕКЦИЯ] Расчет стоимости: {calculationType}, Базовая: {baseCost}, Итоговая: {finalCost}");
    }

    private void OnNotificationSent(string message, string recipient, DateTime timestamp)
    {
        Console.WriteLine($"[КОЛЛЕКЦИЯ] Уведомление: {message}, Получатель: {recipient}, Время: {timestamp:HH:mm:ss}");
    }

    public List<T> Filter(DeliveryFilter filter)
    {
        return _items.Where(item => filter(item)).ToList();
    }

    public void ForEach(Action<T> action)
    {
        _items.ForEach(action);
    }

    public List<T> SortByCost(bool ascending = true)
    {
        return ascending ? 
            _items.OrderBy(item => item.Cost).ToList() : 
            _items.OrderByDescending(item => item.Cost).ToList();
    }

    public Dictionary<string, List<T>> GroupByRegion()
    {
        var grouped = new Dictionary<string, List<T>>();
        foreach (var item in _items)
        {
            if (item is StandardDelivery std)
            {
                var region = std.DeliveryRegion;
                if (!grouped.ContainsKey(region))
                    grouped[region] = new List<T>();
                grouped[region].Add(item);
            }
        }
        return grouped;
    }

    public T FindItem(string id)
    {
        return _items.Find(x => x.DeliveryOptionId == id);
    }

    public void DisplayAll()
    {
        Console.WriteLine($"\n=== Коллекция {typeof(T).Name} ({_items.Count} элементов) ===");
        foreach (var item in _items)
        {
            Console.WriteLine(item.GetDeliveryDetails());
        }
    }

    public void DisplayByStatus(bool isActive)
    {
        Console.WriteLine($"\n=== Опции со статусом {(isActive ? "Активна" : "Неактивна")} ===");
        foreach (var item in _items)
        {
            if (item.IsActive == isActive)
            {
                Console.WriteLine(item.GetDeliveryDetails());
            }
        }
    }

    public void DisplayByCost(decimal minCost, decimal maxCost)
    {
        Console.WriteLine($"\n=== Опции со стоимостью от {minCost} до {maxCost} руб. ===");
        foreach (var item in _items)
        {
            if (item.Cost >= minCost && item.Cost <= maxCost)
            {
                Console.WriteLine(item.GetDeliveryDetails());
            }
        }
    }
}

Console.WriteLine("=== ДЕМОНСТРАЦИЯ РАСШИРЕННЫХ ВОЗМОЖНОСТЕЙ ===\n");

var container = new DependencyContainer();
var emailService = new EmailNotificationService("smtp.myserver.com", 587);
var discountService = new SeasonalDiscountService(0.15m);

container.Register<INotificationService>(emailService);
container.Register<IDiscountService>(discountService);

var deliveryCollection = new DeliveryCollection<ShippingOption>();
deliveryCollection.CollectionChanged += (action, count) => 
{
    Console.WriteLine($"[КОЛЛЕКЦИЯ] Действие: {action}, Всего элементов: {count}");
};

var standard = new StandardDelivery("1", "Стандарт+", 300, 5, 
    container.GetService<INotificationService>(),
    container.GetService<IDiscountService>(),
    50, "Центральный регион");

var express = new ExpressDelivery("2", "Экспресс-24", 500, 1, 150, true);
var pickup = new Pickup("3", "Самовывоз Центр", "ул. Центральная, 1", 14);

deliveryCollection.AddItem(standard);
deliveryCollection.AddItem(express);
deliveryCollection.AddItem(pickup);

Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ НОВЫХ АТРИБУТОВ И МЕТОДОВ ===");

standard.UpdateSupportedCountries(new List<string> { "Россия", "Беларусь", "Казахстан" });
standard.ApplyWeightSurcharge(55);
standard.AddHandlingInstruction("Осторожно, хрупкое содержимое");
standard.SetEcoFriendlyMode(true);
standard.ChangeSecurityLevel("High");

Console.WriteLine($"\nПоддерживается доставка в Германию: {standard.IsCountrySupported("Германия")}");
Console.WriteLine($"Требуется специальная обработка: {standard.RequiresSpecialHandling()}");

express.DisplayAvailableSlots();
express.BookDeliverySlot(DateTime.Now.AddDays(1).Date.AddHours(10));

Console.WriteLine($"\nОкно доставки: {express.DeliveryTimeWindow}");
Console.WriteLine($"Приоритетная загрузка: {express.HasPriorityLoading}");

pickup.AddPaymentMethod("Мобильный кошелек");
pickup.UpdateFacilityInfo(150, false);
Console.WriteLine($"Поддерживается оплата картой: {pickup.SupportsPaymentMethod("Карта")}");

Console.WriteLine("\n=== РАБОТА С КОЛЛЕКЦИЕЙ И ФИЛЬТРАЦИЯ ===");

var expensiveDeliveries = deliveryCollection.Filter(d => d.Cost > 200);
Console.WriteLine($"Дорогие доставки (>200 руб.): {expensiveDeliveries.Count}");

var activeDeliveries = deliveryCollection.Filter(d => d.IsActive);
Console.WriteLine($"Активные доставки: {activeDeliveries.Count}");

var sortedByCost = deliveryCollection.SortByCost();
Console.WriteLine("\nДоставки отсортированные по стоимости:");
sortedByCost.ForEach(d => Console.WriteLine($"  - {d.DeliveryOptionName}: {d.Cost} руб."));

var groupedByRegion = deliveryCollection.GroupByRegion();
Console.WriteLine("\nГруппировка по регионам:");
foreach (var group in groupedByRegion)
{
    Console.WriteLine($"  Регион '{group.Key}': {group.Value.Count} доставок");
}

Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ СОБЫТИЙ ===");
standard.ActivateOption();
standard.UpdateCost(350, "Обновление тарифов");
express.DeactivateOption();

Console.WriteLine("\n=== ПОЛНЫЙ СПИСОК ДОСТАВОК ===");
deliveryCollection.DisplayAll();

Console.WriteLine("\n=== РАСШИРЕННАЯ ИНФОРМАЦИЯ ===");
deliveryCollection.ForEach(d => 
{
    Console.WriteLine($"\n{d.DeliveryOptionName}:");
    Console.WriteLine($"  Статус: {d.GetStatus()}");
    Console.WriteLine($"  Безопасность: {d.GetSecurityInfo()}");
    if (d is StandardDelivery std)
    {
        Console.WriteLine($"  Инструкции: {string.Join("; ", std.SpecialHandlingInstructions)}");
    }
    if (d is Pickup pk)
    {
        Console.WriteLine($"  Способы оплаты: {string.Join(", ", pk.PaymentMethods)}");
        Console.WriteLine($"  Работает сейчас: {pk.IsWorkingNow()}");
    }
});

Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА ===");

=== ДЕМОНСТРАЦИЯ РАСШИРЕННЫХ ВОЗМОЖНОСТЕЙ ===

Добавлено в коллекцию: Стандарт+
[КОЛЛЕКЦИЯ] Действие: Item Added, Всего элементов: 1
Добавлено в коллекцию: Экспресс-24
[КОЛЛЕКЦИЯ] Действие: Item Added, Всего элементов: 2
Добавлено в коллекцию: Самовывоз Центр
[КОЛЛЕКЦИЯ] Действие: Item Added, Всего элементов: 3

=== ДЕМОНСТРАЦИЯ НОВЫХ АТРИБУТОВ И МЕТОДОВ ===
Обновлены поддерживаемые страны: Россия, Беларусь, Казахстан
[КОЛЛЕКЦИЯ] Уведомление: Обновлены поддерживаемые страны для Стандарт+, Получатель: System Administrator, Время: 23:01:30
Применена доплата за вес: 50 руб.
[КОЛЛЕКЦИЯ] Расчет стоимости: Weight Surcharge, Базовая: 300, Итоговая: 350
Добавлена инструкция по обработке: Осторожно, хрупкое содержимое
[КОЛЛЕКЦИЯ] Уведомление: Добавлена инструкция для Стандарт+: Осторожно, хрупкое содержимое, Получатель: Logistics Team, Время: 23:01:30
Эко-режим активирован для доставки Стандарт+
Добавлена инструкция по обработке: Использовать экологичную упаковку
[КОЛЛЕКЦИЯ] Уведомление: Добавл